In [8]:
import time
import pandas as pd
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
import random
from urllib.parse import unquote
import os

if __name__ == '__main__':
    # ตั้งค่า Undetected Chromedriver
    options = uc.ChromeOptions()
    # options.add_argument('--headless')  # เปิดบรรทัดนี้ถ้าไม่ต้องการให้เด้งหน้าต่าง Chrome ขึ้นมา
    
    print("กำลังเปิด Browser...")
    driver = uc.Chrome(options=options, version_main=142)
    
    # ชื่อไฟล์ที่จะบันทึก (เปลี่ยนชื่อไฟล์ใหม่ให้สอดคล้องกับงาน)
    output_filename = 'ddproperty_condo_last_30days_100pages.csv'
    
    # สร้าง Set ไว้เก็บ URL ที่เคยเจอแล้ว เพื่อป้องกันประกาศซ้ำ (Duplicate Listing)
    visited_urls = set()

    try:
        print(f"\n{'='*50}")
        print(f"เริ่มดึงข้อมูลรวม (Global Search) จำนวน 100 หน้า")
        print(f"{'='*50}")

        # --- Loop: วนหน้า 1 ถึง 100 ---
        for page_num in range(1, 101):
            
            # เก็บข้อมูลของหน้านี้
            page_data = []
            
            print(f"\n--- กำลังประมวลผลหน้าที่ {page_num} / 100 ---")
            
            # สร้าง URL ตาม Pattern ที่กำหนด
            target_url = f"https://www.ddproperty.com/รวมประกาศขาย?listingType=sale&page={page_num}&propertyTypeGroup=N&propertyTypeCode=CONDO&isCommercial=false&lastPosted=7"
            
            print(f"URL: {target_url}")
            driver.get(target_url)
            
            # Random sleep เล็กน้อยให้ดูเหมือนคน และรอโหลด
            time.sleep(random.uniform(6, 9))
            
            # --- Check Redirect ---
            # เช็คว่าระบบดีดกลับหน้า 1 หรือไม่ (กรณีหน้าเกินจำนวนที่มีจริง)
            current_url = unquote(driver.current_url)
            if page_num > 1 and "page=1" in current_url and f"page={page_num}" not in current_url:
                 print(f"!!! ถูก Redirect กลับหน้า 1 (คาดว่าหมดหน้าข้อมูลแล้ว) -> จบการทำงาน")
                 break

            # Scroll Down เพื่อให้รูปภาพและข้อมูลโหลด (Lazy Load)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
            time.sleep(2)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.8);")
            time.sleep(2)
            
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            listings = soup.find_all('a', attrs={'class': 'card-footer'}) 

            if len(listings) == 0:
                print(f"ไม่พบประกาศขายในหน้านี้ -> จบการทำงาน")
                break

            # --- Extract Data Loop ---
            new_item_count = 0
            
            for item in listings:
                row = {}
                
                # URL
                url_raw = item.get('href')
                if url_raw and not url_raw.startswith('http'):
                     url_raw = "https://www.ddproperty.com" + url_raw
                
                row['url'] = url_raw

                # Check Duplicate Logic
                if row['url'] not in visited_urls:
                    new_item_count += 1
                    visited_urls.add(row['url'])
                    
                    # Extract Data Helper Function
                    def get_text_by_da_id(da_id):
                        el = item.find(attrs={'da-id': da_id}) 
                        return el.get_text(strip=True) if el else "N/A"

                    title_el = item.find(class_='title-badge-wrapper')
                    row['title'] = title_el.get_text(strip=True) if title_el else "N/A"
                    
                    row['publish_date'] = get_text_by_da_id('listing-card-v2-recency')
                    row['price'] = get_text_by_da_id('listing-card-v2-price')
                    row['price_per_sqm'] = get_text_by_da_id('listing-card-v2-psf')
                    row['usable_area'] = get_text_by_da_id('listing-card-v2-area')
                    row['bedroom'] = get_text_by_da_id('listing-card-v2-bedrooms')
                    row['restroom'] = get_text_by_da_id('listing-card-v2-bathrooms')
                    
                    address_el = item.find(class_='listing-address')
                    row['full_address'] = address_el.get_text(strip=True) if address_el else "N/A"
                    
                    # field ที่ไม่มีในการค้นหาแบบรวม แต่ใส่ไว้ให้ format ตรงกัน
                    row['coords'] = "-"
                    
                    page_data.append(row)

            print(f"เจอทั้งหมด {len(listings)} รายการ | เป็นรายการใหม่ {new_item_count} รายการ")
            
            if new_item_count == 0 and len(listings) > 0:
                print("Note: หน้าข้อมูลซ้ำเดิมทั้งหมด (อาจเป็น Featured Listings)")
                # ไม่ break เพราะหน้าถัดไปอาจมีของใหม่

            # --- Save Data Per Page (บันทึกทีละหน้า กันโปรแกรมหลุด) ---
            if page_data:
                df = pd.DataFrame(page_data)
                
                # จัดเรียงคอลัมน์
                cols = ['url', 'title', 'publish_date', 'price', 
                        'price_per_sqm', 'usable_area', 'bedroom', 'restroom', 'coords', 'full_address']
                
                # เลือกเฉพาะคอลัมน์ที่มีจริง
                existing_cols = [c for c in cols if c in df.columns]
                df = df[existing_cols]

                # ถ้าไฟล์ยังไม่มี ให้เขียน Header, ถ้ามีแล้วให้ Append
                use_header = not os.path.exists(output_filename)
                df.to_csv(output_filename, mode='a', index=False, encoding='utf-8-sig', header=use_header)
                print(f"--> บันทึก {len(df)} รายการ ลงไฟล์เรียบร้อย")
            
            # พักก่อนไปหน้าถัดไป
            sleep_time = random.uniform(3, 5)
            print(f"พัก {sleep_time:.2f} วินาที...")
            time.sleep(sleep_time)

    except Exception as e:
        print(f"เกิดข้อผิดพลาดร้ายแรง: {e}")

    finally:
        print("ปิด Browser...")
        driver.quit()
        print(f"เสร็จสิ้น! ข้อมูลถูกบันทึกที่: {output_filename}")

กำลังเปิด Browser...

เริ่มดึงข้อมูลรวม (Global Search) จำนวน 100 หน้า

--- กำลังประมวลผลหน้าที่ 1 / 100 ---
URL: https://www.ddproperty.com/รวมประกาศขาย?listingType=sale&page=1&propertyTypeGroup=N&propertyTypeCode=CONDO&isCommercial=false&lastPosted=7
ปิด Browser...
เสร็จสิ้น! ข้อมูลถูกบันทึกที่: ddproperty_condo_last_30days_100pages.csv


KeyboardInterrupt: 

In [ ]:
import time
import pandas as pd
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
import random
from urllib.parse import unquote
import os
import sys

# --- Configuration ---
CONFIG = {
    'chrome_version': 142,       # เปลี่ยนให้ตรงกับ Chrome ในเครื่องคุณ
    'output_file': 'ddproperty_condo_last_7days_pipeline.csv',
    'max_pages': 100,
    'sleep_range': (6, 9),       # ช่วงเวลาสุ่มรอ (วินาที)
    'base_url': "https://www.ddproperty.com/รวมประกาศขาย?listingType=sale&propertyTypeGroup=N&propertyTypeCode=CONDO&isCommercial=false&lastPosted=7"
}

class DDPropertyPipeline:
    def __init__(self, config):
        self.config = config
        self.driver = None
        self.visited_urls = set()
        
    def _init_driver(self):
        """Step 1: เริ่มต้น Browser"""
        print("[Pipeline] Initializing Browser...")
        options = uc.ChromeOptions()
        # options.add_argument('--headless') # อย่าเปิดถ้าติด Cloudflare
        
        try:
            self.driver = uc.Chrome(
                options=options, 
                version_main=self.config['chrome_version'],
                use_subprocess=True
            )
        except Exception as e:
            print(f"[Error] Driver Init Failed: {e}")
            sys.exit(1)

    def _human_verification_checkpoint(self):
        """Step 2: จุดพักให้คนกดยืนยันตัวตน (Cloudflare)"""
        print("\n" + "="*50)
        print("!!! HUMAN VERIFICATION CHECKPOINT !!!")
        print("ระบบจะเปิดหน้าแรก... กรุณาแก้ Cloudflare (ติ๊กถูก) ให้เรียบร้อย")
        print("="*50)
        
        # เปิดหน้าแรกเพื่อให้ User เคลียร์ Cloudflare
        self.driver.get("https://www.ddproperty.com")
        
        input(">>> เมื่อเข้าหน้าเว็บได้ปกติแล้ว ให้กด Enter ที่นี่เพื่อเริ่มดูดข้อมูล... ")
        print("[Pipeline] Starting extraction process...\n")

    def _scroll_page(self):
        """Helper: เลื่อนหน้าจอเพื่อให้รูปและข้อมูลโหลด (Lazy Load)"""
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        time.sleep(2)
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.85);")
        time.sleep(2)

    def _extract_data_from_html(self, soup):
        """Step 4: แปลง HTML Soup เป็น List of Dictionaries"""
        listings = soup.find_all('a', attrs={'class': 'card-footer'})
        extracted_rows = []
        new_count = 0

        for item in listings:
            row = {}
            
            # Get URL
            url_raw = item.get('href')
            if url_raw and not url_raw.startswith('http'):
                url_raw = "https://www.ddproperty.com" + url_raw
            row['url'] = url_raw

            # Check Duplicates
            if row['url'] in self.visited_urls:
                continue

            self.visited_urls.add(row['url'])
            new_count += 1

            # Helper extraction
            def get_text(da_id):
                el = item.find(attrs={'da-id': da_id})
                return el.get_text(strip=True) if el else "N/A"

            # Parse Fields
            title_el = item.find(class_='title-badge-wrapper')
            row['title'] = title_el.get_text(strip=True) if title_el else "N/A"
            row['publish_date'] = get_text('listing-card-v2-recency')
            row['price'] = get_text('listing-card-v2-price')
            row['price_per_sqm'] = get_text('listing-card-v2-psf')
            row['usable_area'] = get_text('listing-card-v2-area')
            row['bedroom'] = get_text('listing-card-v2-bedrooms')
            row['restroom'] = get_text('listing-card-v2-bathrooms')
            
            address_el = item.find(class_='listing-address')
            row['full_address'] = address_el.get_text(strip=True) if address_el else "N/A"
            
            # Placeholders
            row['floor'] = "-"
            row['coords'] = "-"

            extracted_rows.append(row)
        
        return extracted_rows, len(listings), new_count

    def _save_batch(self, data):
        """Step 5: บันทึกข้อมูลลง CSV (Append mode)"""
        if not data:
            return

        df = pd.DataFrame(data)
        cols = ['url', 'title', 'publish_date', 'price', 'price_per_sqm', 
                'usable_area', 'bedroom', 'restroom', 'coords', 'full_address']
        
        # Filter existing columns
        existing_cols = [c for c in cols if c in df.columns]
        df = df[existing_cols]

        filename = self.config['output_file']
        use_header = not os.path.exists(filename)
        
        try:
            df.to_csv(filename, mode='a', index=False, encoding='utf-8-sig', header=use_header)
            print(f"   [Save] Saved {len(df)} new records.")
        except Exception as e:
            print(f"   [Error] Save failed: {e}")

    def run(self):
        """Main Execution Flow"""
        self._init_driver()
        
        try:
            self._human_verification_checkpoint()

            for page in range(1, self.config['max_pages'] + 1):
                print(f"--- Processing Page {page} / {self.config['max_pages']} ---")
                
                # Construct URL
                target_url = f"{self.config['base_url']}&page={page}"
                self.driver.get(target_url)
                
                # Sleep (Human behavior)
                sleep_time = random.uniform(*self.config['sleep_range'])
                time.sleep(sleep_time)

                # Check Redirect (End of pages)
                if page > 1 and "page=1" in unquote(self.driver.current_url) and f"page={page}" not in unquote(self.driver.current_url):
                    print("   [Info] Redirected to page 1. Assuming end of listings.")
                    break
                
                # Step 3: Extract & Scroll
                self._scroll_page()
                soup = BeautifulSoup(self.driver.page_source, 'html.parser')
                
                # Step 4: Transform
                data, total_found, new_found = self._extract_data_from_html(soup)
                print(f"   [Stats] Found: {total_found} | New: {new_found}")

                if total_found == 0:
                    print("   [Info] No listings found on this page. Stopping.")
                    break

                # Step 5: Load
                self._save_batch(data)
                
        except KeyboardInterrupt:
            print("\n[Pipeline] Stopped by user.")
        except Exception as e:
            print(f"\n[Pipeline] Critical Error: {e}")
        finally:
            print("[Pipeline] Closing Browser...")
            if self.driver:
                self.driver.quit()
            print(f"[Pipeline] Finished. Data saved to: {self.config['output_file']}")

if __name__ == '__main__':
    pipeline = DDPropertyPipeline(CONFIG)
    pipeline.run()

In [ ]:
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import time
import pandas as pd
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
import random
from urllib.parse import unquote
import os
import sys

# --- CONFIGURATION ---
# Path ที่ต้องการเซฟไฟล์ (ควรเป็น Path ที่ Airflow Worker เข้าถึงได้ หรือ Mounted Volume)
OUTPUT_PATH = "/opt/airflow/dags/data/ddproperty_data.csv" 

CONFIG = {
    'chrome_version': 131,       # Update ให้ตรงกับ Chrome ใน Docker/Server ของ Airflow
    'output_file': OUTPUT_PATH,
    'max_pages': 100,
    'sleep_range': (10, 15),     # เพิ่มเวลาพักให้นานขึ้น เพื่อลดโอกาสโดนจับเมื่อรัน Headless
    'base_url': "https://www.ddproperty.com/รวมประกาศขาย?listingType=sale&propertyTypeGroup=N&propertyTypeCode=CONDO&isCommercial=false&lastPosted=7"
}

# --- CLASS DEFINITION (Embedded inside DAG file for simplicity) ---
class DDPropertyPipeline:
    def __init__(self, config):
        self.config = config
        self.driver = None
        self.visited_urls = set()
        
    def _init_driver(self):
        print("[Pipeline] Initializing Browser (Headless Mode)...")
        options = uc.ChromeOptions()
        
        # --- AIRFLOW SPECIFIC SETTINGS ---
        # จำเป็นต้องเปิด Headless เมื่อรันบน Server
        options.add_argument('--headless=new') 
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--window-size=1920,1080')
        
        try:
            # หมายเหตุ: version_main อาจต้องปรับตาม Chrome ที่ติดตั้งใน Airflow Docker Image
            self.driver = uc.Chrome(
                options=options, 
                version_main=self.config['chrome_version'],
                use_subprocess=True
            )
        except Exception as e:
            print(f"[Error] Driver Init Failed: {e}")
            raise e

    def _scroll_page(self):
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
        time.sleep(2)
        self.driver.execute_script("window.scrollTo(0, document.body.scrollHeight * 0.85);")
        time.sleep(2)

    def _extract_data_from_html(self, soup):
        listings = soup.find_all('a', attrs={'class': 'card-footer'})
        extracted_rows = []
        new_count = 0

        for item in listings:
            row = {}
            url_raw = item.get('href')
            if url_raw and not url_raw.startswith('http'):
                url_raw = "https://www.ddproperty.com" + url_raw
            row['url'] = url_raw

            if row['url'] in self.visited_urls:
                continue

            self.visited_urls.add(row['url'])
            new_count += 1

            def get_text(da_id):
                el = item.find(attrs={'da-id': da_id})
                return el.get_text(strip=True) if el else "N/A"

            title_el = item.find(class_='title-badge-wrapper')
            row['title'] = title_el.get_text(strip=True) if title_el else "N/A"
            row['publish_date'] = get_text('listing-card-v2-recency')
            row['price'] = get_text('listing-card-v2-price')
            row['price_per_sqm'] = get_text('listing-card-v2-psf')
            row['usable_area'] = get_text('listing-card-v2-area')
            row['bedroom'] = get_text('listing-card-v2-bedrooms')
            row['restroom'] = get_text('listing-card-v2-bathrooms')
            
            address_el = item.find(class_='listing-address')
            row['full_address'] = address_el.get_text(strip=True) if address_el else "N/A"
            
            row['floor'] = "-"
            row['coords'] = "-"

            extracted_rows.append(row)
        
        return extracted_rows, len(listings), new_count

    def _save_batch(self, data):
        if not data:
            return

        df = pd.DataFrame(data)
        cols = ['url', 'title', 'publish_date', 'price', 'price_per_sqm', 
                'usable_area', 'bedroom', 'restroom', 'coords', 'full_address']
        
        existing_cols = [c for c in cols if c in df.columns]
        df = df[existing_cols]

        # Ensure directory exists
        os.makedirs(os.path.dirname(self.config['output_file']), exist_ok=True)
        
        filename = self.config['output_file']
        use_header = not os.path.exists(filename)
        
        try:
            df.to_csv(filename, mode='a', index=False, encoding='utf-8-sig', header=use_header)
            print(f"   [Save] Saved {len(df)} new records to {filename}")
        except Exception as e:
            print(f"   [Error] Save failed: {e}")

    def run(self):
        self._init_driver()
        try:
            # ตัด Human Checkpoint ออก เพราะ Airflow ไม่มีคนกด
            print("WARNING: Running in Automated Mode (No Human Verification). Cloudflare might block this.")
            
            for page in range(1, self.config['max_pages'] + 1):
                print(f"--- Processing Page {page} / {self.config['max_pages']} ---")
                
                target_url = f"{self.config['base_url']}&page={page}"
                self.driver.get(target_url)
                
                # Sleep นานขึ้นในโหมด Headless
                sleep_time = random.uniform(*self.config['sleep_range'])
                time.sleep(sleep_time)

                # Check Title for Cloudflare Block
                if "Just a moment" in self.driver.title or "Security" in self.driver.title:
                    print("!!! BLOCKED BY CLOUDFLARE !!!")
                    raise Exception("Cloudflare blocking detected. Aborting task.")

                # Check Redirect
                if page > 1 and "page=1" in unquote(self.driver.current_url) and f"page={page}" not in unquote(self.driver.current_url):
                    print("   [Info] Redirected to page 1. Assuming end of listings.")
                    break
                
                self._scroll_page()
                soup = BeautifulSoup(self.driver.page_source, 'html.parser')
                
                data, total_found, new_found = self._extract_data_from_html(soup)
                print(f"   [Stats] Found: {total_found} | New: {new_found}")

                if total_found == 0:
                    print("   [Info] No listings found. Stopping.")
                    break

                self._save_batch(data)
                
        except Exception as e:
            print(f"\n[Pipeline] Critical Error: {e}")
            raise e # Raise error to mark Task as Failed in Airflow
        finally:
            print("[Pipeline] Closing Browser...")
            if self.driver:
                self.driver.quit()

# --- AIRFLOW DAG DEFINITION ---

def run_ddproperty_scraper():
    pipeline = DDPropertyPipeline(CONFIG)
    pipeline.run()

default_args = {
    'owner': 'airflow',
    'depends_on_past': False,
    'email_on_failure': False,
    'email_on_retry': False,
    'retries': 1,
    'retry_delay': timedelta(minutes=5),
}

with DAG(
    'ddproperty_scraper_pipeline',
    default_args=default_args,
    description='Scrape Condo Data from DDProperty',
    schedule_interval='@daily', # รันทุกวัน
    start_date=datetime(2023, 1, 1),
    catchup=False,
    tags=['scraping', 'selenium'],
) as dag:

    scrape_task = PythonOperator(
        task_id='scrape_ddproperty_task',
        python_callable=run_ddproperty_scraper,
    )

    scrape_task